# 🏆 Full Pipeline — Độ Chính Xác Cao Nhất

## Pipeline 3 bước:
1. **Pretrain trên FER2013** (~60MB, nhanh) → học facial features
2. **Fine-tune trên DAiSEE** (~15GB) → chuyên biệt hóa cho lớp học
3. **Tối ưu**: EfficientNetB0 + Focal Loss + Class Weights mạnh

**Kết quả kỳ vọng:** val_accuracy ~70-80% (so với 56% cũ)

> ⚡ **Bật GPU trước:** Runtime → Change runtime type → GPU (T4 hoặc A100)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — Kiểm tra GPU & Cài thư viện              ║
# ╚══════════════════════════════════════════════════════╝
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for l in result.stdout.split('\n')[:4]: print(l)
else:
    print('❌ KHÔNG có GPU! Vào Runtime → Change runtime type → GPU')

!pip install -q kaggle opencv-python-headless scikit-learn seaborn
print('✅ Thư viện sẵn sàng!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — Kết nối Kaggle API                       ║
# ╚══════════════════════════════════════════════════════╝
import os
from google.colab import files

print('📁 Upload file kaggle.json:')
uploaded = files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API sẵn sàng!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — Tải FER2013 (~60MB, rất nhanh)           ║
# ╚══════════════════════════════════════════════════════╝
print('⏳ Đang tải FER2013...')
!kaggle datasets download -d msambare/fer2013 -p /content/fer2013 --unzip
print('✅ FER2013 đã tải!')
!find /content/fer2013 -maxdepth 2 -type d

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — Chuẩn bị FER2013 & Pretrain              ║
# ╚══════════════════════════════════════════════════════╝
import shutil
import numpy as np
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# --- Ánh xạ nhãn FER2013 (7 lớp) → 4 nhãn hệ thống ---
# Mapping: angry/disgust/fear/sad → Buồn chán(0)
#          neutral → Bình thường(3)
#          happy/surprise → Hứng thú(2)
#          (Tập trung không có trong FER → sẽ học từ DAiSEE)
FER_MAPPING = {
    'angry': 0, 'disgust': 0, 'fear': 0, 'sad': 0,
    'neutral': 3,
    'happy': 2, 'surprise': 2,
}
EMOTION_NAMES = ['Buon_chan', 'Tap_trung', 'Hung_thu', 'Binh_thuong']
OUTPUT_FER = Path('/content/fer_mapped')
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
NUM_CLASSES = 4

for split in ['train', 'test']:
    for i in range(4):
        (OUTPUT_FER / split / str(i)).mkdir(parents=True, exist_ok=True)

fer_base = Path('/content/fer2013')
splits = {}
for p in fer_base.rglob('*'):
    if p.is_dir() and p.name.lower() in ['train', 'training']:
        splits['train'] = p
    elif p.is_dir() and p.name.lower() in ['test', 'validation', 'val']:
        splits['test'] = p

count = 0
for split_name, split_path in splits.items():
    for cls_dir in split_path.iterdir():
        if not cls_dir.is_dir(): continue
        new_label = FER_MAPPING.get(cls_dir.name.lower(), 3)
        for img_path in list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')):
            shutil.copy2(img_path, OUTPUT_FER / split_name / str(new_label) / img_path.name)
            count += 1

print(f'✅ Đã ánh xạ {count:,} ảnh FER2013!')
for split in ['train', 'test']:
    print(f'  {split}:')
    for i, name in enumerate(EMOTION_NAMES):
        n = len(list((OUTPUT_FER/split/str(i)).glob('*.*')))
        print(f'    [{i}] {name}: {n:,} ảnh')

# --- Data Generators ---
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.7, 1.3],
    shear_range=0.1
)
val_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_directory(
    str(OUTPUT_FER/'train'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_flow = val_gen.flow_from_directory(
    str(OUTPUT_FER/'test'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# --- Class Weights (tăng mạnh cho lớp thiểu số) ---
classes = np.unique(train_flow.classes)
cw = compute_class_weight('balanced', classes=classes, y=train_flow.classes)
cw_dict = {c: w for c, w in zip(classes, cw)}
if 0 in cw_dict: cw_dict[0] *= 2.0  # Buồn chán
if 3 in cw_dict: cw_dict[3] *= 3.0  # Bình thường
if 1 not in cw_dict: cw_dict[1] = 1.0 # Tránh lỗi thiếu class 
print('⚖️  Class weights:', {EMOTION_NAMES[k]: round(v,2) for k,v in cw_dict.items()})

# --- Build EfficientNetB0 (tốt hơn MobileNetV2) ---
base = EfficientNetB0(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(base.input, out)

model.compile(
    optimizer=Adam(1e-3),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

os.makedirs('/content/checkpoints', exist_ok=True)

print('\n🚀 Bắt đầu Pretrain trên FER2013...')
history_fer = model.fit(
    train_flow, validation_data=val_flow,
    epochs=15,
    class_weight=cw_dict,
    callbacks=[
        ModelCheckpoint('/content/checkpoints/fer_pretrain.h5',
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]
)
best_fer = max(history_fer.history['val_accuracy'])
print(f'\n✅ FER2013 Pretrain xong! Best val_acc = {best_fer:.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 5 — Tải DAiSEE (~15GB, mất 10-15 phút)      ║
# ╚══════════════════════════════════════════════════════╝
print('⏳ Đang tải DAiSEE (~15GB)...')
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip
print('✅ DAiSEE đã tải!')
!find /content/daisee_raw -maxdepth 3 -type d | head -20

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 — Chuẩn bị & Trích xuất Frame DAiSEE      ║
# ╚══════════════════════════════════════════════════════╝
import pandas as pd
import cv2
import glob
from tqdm.notebook import tqdm

EMOTION_NAMES_VN = ['Buồn chán', 'Tập trung', 'Hứng thú', 'Bình thường']

csv_files = glob.glob('/content/daisee_raw/**/*.csv', recursive=True)
print('📄 CSV files tìm thấy:', csv_files)

def find_csv(name):
    matches = [f for f in csv_files if name.lower() in f.lower()]
    return matches[0] if matches else None

df_train = pd.read_csv(find_csv('train'))
df_val   = pd.read_csv(find_csv('validat'))
df_test  = pd.read_csv(find_csv('test'))

def map_label(row):
    b = row.get('Boredom', row.get('boredom', 0))
    e = row.get('Engagement', row.get('engagement', 0))
    c = row.get('Confusion', row.get('confusion', 0))
    f = row.get('Frustration', row.get('frustration', 0))
    if b >= 2 and e <= 1:           return 0  # Buồn chán
    if e >= 2 and c >= 1 and f < 2: return 2  # Hứng thú
    if e >= 2:                      return 1  # Tập trung
    return 3                                   # Bình thường

for df, name in [(df_train,'Train'), (df_val,'Val'), (df_test,'Test')]:
    df['emotion_label'] = df.apply(map_label, axis=1)
    print(f'{name}: {len(df):,} clips | Distribution: {df["emotion_label"].value_counts().to_dict()}')

OUTPUT_DIR = Path('/content/frames')
FRAMES_PER_CLIP = 8  # Tăng lên 8 để có nhiều data hơn

for split in ['train', 'val', 'test']:
    for i in range(4):
        (OUTPUT_DIR / split / str(i)).mkdir(parents=True, exist_ok=True)

def find_video(clip_name):
    for p in [f'/content/daisee_raw/**/{clip_name}',
              f'/content/daisee_raw/**/{clip_name}.avi']:
        found = glob.glob(p, recursive=True)
        if found: return found[0]
    return None

def extract_frames(video_path, label, split, clip_id):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release(); return 0
    indices = np.linspace(0, total-1, FRAMES_PER_CLIP, dtype=int)
    saved = 0
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        frame = cv2.resize(frame, IMG_SIZE)
        cv2.imwrite(str(OUTPUT_DIR/split/str(label)/f'{clip_id}_{i}.jpg'), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 92])
        saved += 1
    cap.release()
    return saved

stats = {'train':0, 'val':0, 'test':0}
for df, split in [(df_train,'train'), (df_val,'val'), (df_test,'test')]:
    print(f'\n⏳ Đang xử lý {split}...')
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        clip_name = str(row.iloc[0])
        label = int(row['emotion_label'])
        video_path = find_video(clip_name)
        if video_path is None: continue
        stats[split] += extract_frames(video_path, label, split, idx)

print('\n✅ Trích xuất xong!')
for s, n in stats.items():
    print(f'  {s}: {n:,} frames')
print('\nPhân phối sau trích xuất:')
for split in ['train', 'val', 'test']:
    print(f'  {split}:')
    for i, name in enumerate(EMOTION_NAMES_VN):
        n = len(list((OUTPUT_DIR/split/str(i)).glob('*.jpg')))
        print(f'    [{i}] {name}: {n:,} frames')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Fine-tune trên DAiSEE (Bước quan trọng nhất)    ║
# ╚══════════════════════════════════════════════════════════════╝
from tensorflow.keras.models import load_model
from tensorflow.keras.regularizers import l2

# Load pretrained model từ FER2013
print('📦 Loading pretrained model từ FER2013...')
pretrained = load_model('/content/checkpoints/fer_pretrain.h5')

# --- Tạo Data Generators cho DAiSEE ---
train_gen_d = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.7, 1.3],
    shear_range=0.15
)
val_gen_d = ImageDataGenerator(rescale=1./255)

train_d = train_gen_d.flow_from_directory(
    str(OUTPUT_DIR/'train'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_d = val_gen_d.flow_from_directory(
    str(OUTPUT_DIR/'val'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_d = val_gen_d.flow_from_directory(
    str(OUTPUT_DIR/'test'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# --- Class Weights siêu mạnh cho lớp thiểu số ---
cw_d = compute_class_weight('balanced', classes=np.unique(train_d.classes), y=train_d.classes)
cw_d_dict = dict(enumerate(cw_d))
# Tăng mạnh trọng số cho lớp rất ít dữ liệu
cw_d_dict[0] *= 4.0  # Buồn chán (rất ít)
cw_d_dict[3] *= 6.0  # Bình thường (ít nhất)
print('⚖️  DAiSEE Class weights:', {EMOTION_NAMES_VN[k]: round(v,2) for k,v in cw_d_dict.items()})

# ====== PHASE 1: Chỉ train classification head ======
# Đóng băng base model, chỉ mở classification layers
for layer in pretrained.layers[:-6]:
    layer.trainable = False

pretrained.compile(
    optimizer=Adam(1e-3),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print('\n🔥 Phase 1 — Training head trên DAiSEE...')
hist1 = pretrained.fit(
    train_d, validation_data=val_d,
    epochs=10,
    class_weight=cw_d_dict,
    callbacks=[
        ModelCheckpoint('/content/checkpoints/daisee_phase1.h5',
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]
)
print(f'Phase 1 xong! Best val_acc = {max(hist1.history["val_accuracy"]):.4f}')

# ====== PHASE 2: Fine-tune 50 layer cuối ======
for layer in pretrained.layers:
    layer.trainable = True
# Đóng băng 60% đầu
freeze_until = int(len(pretrained.layers) * 0.6)
for layer in pretrained.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in pretrained.layers if l.trainable)
print(f'\n🔓 Fine-tune {trainable_count} layers cuối...')

pretrained.compile(
    optimizer=Adam(1e-4),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print('🔥 Phase 2 — Fine-tuning trên DAiSEE...')
hist2 = pretrained.fit(
    train_d, validation_data=val_d,
    epochs=30,
    class_weight=cw_d_dict,
    callbacks=[
        ModelCheckpoint('/content/checkpoints/daisee_phase2.h5',
                        monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1)
    ]
)
print(f'Phase 2 xong! Best val_acc = {max(hist2.history["val_accuracy"]):.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — Đánh giá toàn diện                      ║
# ╚══════════════════════════════════════════════════════╝
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

best_model = load_model('/content/checkpoints/daisee_phase2.h5')

print('🔍 Đánh giá trên Test Set...')
test_d.reset()
y_pred = best_model.predict(test_d, verbose=1)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true = test_d.classes[:len(y_pred_cls)]

print('\n📊 Classification Report:')
print(classification_report(y_true, y_pred_cls, target_names=EMOTION_NAMES_VN))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_NAMES_VN, yticklabels=EMOTION_NAMES_VN, ax=axes[0])
axes[0].set_title('Confusion Matrix — Test Set', fontsize=14)
axes[0].set_ylabel('Thực tế'); axes[0].set_xlabel('Dự đoán')

# Training curves
all_acc = hist1.history['accuracy'] + hist2.history['accuracy']
all_val  = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
ep = range(1, len(all_acc)+1)
axes[1].plot(ep, all_acc, label='Train Acc', color='royalblue')
axes[1].plot(ep, all_val, label='Val Acc', color='orange')
axes[1].axvline(x=len(hist1.history['accuracy']), color='red',
                linestyle='--', alpha=0.5, label='Fine-tune Start')
axes[1].set_title('Training Accuracy Curve', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/final_results.png', dpi=150)
plt.show()
print('✅ Đánh giá xong!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 9 — Lưu & Tải Model về máy                  ║
# ╚══════════════════════════════════════════════════════╝
import zipfile
from google.colab import files

MODEL_OUTPUT = '/content/emotion_model_daisee.h5'
best_model.save(MODEL_OUTPUT)
print(f'✅ Model đã lưu: {MODEL_OUTPUT}')

with zipfile.ZipFile('/content/emotion_results_v2.zip', 'w') as zf:
    zf.write(MODEL_OUTPUT, 'emotion_model_daisee.h5')
    for f in ['final_results.png']:
        p = f'/content/{f}'
        if os.path.exists(p): zf.write(p, f)

print('⬇️  Đang tải file về máy...')
files.download('/content/emotion_results_v2.zip')
print('\n🏆 HOÀN TẤT!')
print('📌 Giải nén zip, đặt emotion_model_daisee.h5 vào thư mục TTCS1/models/')
print('📌 Chạy: python main.py')